In [1]:
import pandas as pd
import duckdb

df_alarm_log = pd.DataFrame({
    "alarm_id": [
        101, 102, 103, 104, 105,
        201, 202, 203, 204,
        301, 302, 303, 304, 305
    ],
    "device_id": [
        "R05", "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34", "R34"
    ],
    "alarm_time": [
        "2026-08-01 08:00:00",
        "2026-08-01 09:00:00",
        "2026-08-01 10:00:00",
        "2026-08-01 10:00:00",
        "2026-08-01 11:00:00",

        "2026-08-01 08:30:00",
        "2026-08-01 09:30:00",
        "2026-08-01 10:30:00",
        "2026-08-01 11:30:00",

        "2026-08-01 07:00:00",
        "2026-08-01 08:00:00",
        "2026-08-01 09:00:00",
        "2026-08-01 10:00:00",
        "2026-08-01 10:00:00"
    ],
    "alarm_level": [
        "WARNING", "ERROR", "WARNING", "WARNING", "ERROR",
        "ERROR", "WARNING", "ERROR", "WARNING",
        "WARNING", "ERROR", "WARNING", "ERROR", "ERROR"
    ],
    "alarm_value": [
        72, 91, 78, 81, 95,
        88, 70, 93, 76,
        68, 89, 74, 92, 96
    ]
})

df_alarm_log["alarm_time"] = pd.to_datetime(
    df_alarm_log["alarm_time"]
)

df_alarm_log

,alarm_id,device_id,alarm_time,alarm_level,alarm_value
0,101,R05,2026-08-01 08:00:00,WARNING,72
1,102,R05,2026-08-01 09:00:00,ERROR,91
2,103,R05,2026-08-01 10:00:00,WARNING,78
3,104,R05,2026-08-01 10:00:00,WARNING,81
4,105,R05,2026-08-01 11:00:00,ERROR,95
5,201,R16,2026-08-01 08:30:00,ERROR,88
6,202,R16,2026-08-01 09:30:00,WARNING,70
7,203,R16,2026-08-01 10:30:00,ERROR,93
8,204,R16,2026-08-01 11:30:00,WARNING,76
9,301,R34,2026-08-01 07:00:00,WARNING,68


# SQL Daily Review：每台设备每种告警等级的最新记录

## 题目背景

设备运行过程中会产生不同等级的告警记录。

目前主要关注两种告警：

- `WARNING`
- `ERROR`

现在需要查看每台设备在**每一种告警等级下最新的一条记录**。

---

## 题目要求

对于：

```text
device_id + alarm_level
```

每一种组合，找出最新的一条告警记录。

例如：

```text
R05 + WARNING → 最新一条 WARNING
R05 + ERROR   → 最新一条 ERROR
R16 + WARNING → 最新一条 WARNING
R16 + ERROR   → 最新一条 ERROR
```

因此，一台设备最多可以返回两条记录。

### 排名规则

在每个：

```text
device_id + alarm_level
```

分组内部，按照：

1. `alarm_time` 降序；
2. 当 `alarm_time` 相同时，`alarm_id` 降序。

确定最新记录。

---

## 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `alarm_level` | 告警等级 |
| `alarm_id` | 告警编号 |
| `alarm_time` | 告警时间 |
| `alarm_value` | 告警值 |

---

## 最终排序

按照：

1. `device_id` 升序；
2. `alarm_level` 升序。

---

## 解题要求

- 使用 `ROW_NUMBER()`；
- 使用 CTE；
- 每个 `device_id + alarm_level` 组合独立排名；
- 时间相同时使用 `alarm_id` 决定先后；
- 最终每个组合只保留排名第 1 的记录；
- 不使用 `GROUP BY + MAX()`；
- 不使用相关子查询。

## 本题重点

思考窗口函数中的：

```sql
PARTITION BY
```

这一次应该如何定义“一个组”。

不要只因为以前写过：

```sql
PARTITION BY device_id
```

就机械地继续这么写。

本题真正需要回答的是：

> **我到底希望在哪个粒度内部进行排名？**

In [5]:
query = """
WITH rank_table AS (
    SELECT
        device_id,
        alarm_level,
        alarm_id,
        alarm_time,
        alarm_value,
        ROW_NUMBER() OVER (
            PARTITION BY device_id, alarm_level
            ORDER BY alarm_time DESC, alarm_id DESC
        ) AS alarm_rank
    FROM df_alarm_log
)

SELECT
    device_id,
    alarm_level,
    alarm_id,
    alarm_time,
    alarm_value
FROM rank_table
WHERE alarm_rank = 1
ORDER BY
    device_id,
    alarm_level;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,alarm_level,alarm_id,alarm_time,alarm_value
0,R05,ERROR,105,2026-08-01 11:00:00,95
1,R05,WARNING,104,2026-08-01 10:00:00,81
2,R16,ERROR,203,2026-08-01 10:30:00,93
3,R16,WARNING,204,2026-08-01 11:30:00,76
4,R34,ERROR,305,2026-08-01 10:00:00,96
5,R34,WARNING,303,2026-08-01 09:00:00,74
